# BCL-2 Inhibitors — ML Pipeline

Full ML pipeline applied to **BCL-2**. All functions imported from `pipeline/`.

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

from pipeline import (
    load_dataset, check_required_columns,
    first_filter_molecules, remove_duplicate_ids,
    remove_high_ic50_percentage, standardize_molecules,
    calculate_descriptors, calculate_morgan_fingerprints,
    clean_na_and_duplicates, remove_highly_correlated_columns,
    remove_low_variance_columns, standardize_data, merge_ID_response_variable,
    create_binary_activity_median, split_train_test, feature_selection_lasso,
    evaluate_logistic_regression, evaluate_knn, evaluate_gaussian_nb,
    evaluate_bernoulli_nb, evaluate_svm, evaluate_gradient_boosting,
    evaluate_random_forest, evaluate_mlp,
    save_results, save_descriptor_list,
)

## 1. Configuration

In [ ]:
DATA_PATH    = "../data/raw/set_BCL-2.csv"
TARGET_NAME  = "Apoptosis regulator Bcl-2"
TARGET_KEY   = "BCL-2"   # used as results subfolder

IC50_PERCENTILE_REMOVE = 0.05
FP_N_BITS    = 2048
FP_RADIUS    = 2
TEST_SIZE    = 0.2
RANDOM_STATE = 999

## 2. Data loading and filtering

In [ ]:
df = load_dataset(DATA_PATH)
log = pd.DataFrame(columns=["Step", "Removed", "Remaining"])
df, log = first_filter_molecules(df, TARGET_NAME)
df, log = remove_duplicate_ids(df, log)
df, log = remove_high_ic50_percentage(df, IC50_PERCENTILE_REMOVE, log)
df, log = standardize_molecules(df, log=log)
log

## 3. Molecular descriptors

In [ ]:
descriptors = calculate_descriptors(df)
descriptors, log = clean_na_and_duplicates(descriptors, log)
descriptors, _   = remove_highly_correlated_columns(descriptors)
descriptors, _   = remove_low_variance_columns(descriptors)
descriptors      = standardize_data(descriptors)
descriptors      = merge_ID_response_variable(descriptors, df)

## 4. Morgan fingerprints (ECFP4)

In [ ]:
fingerprints, _ = calculate_morgan_fingerprints(
    df, n_bits=FP_N_BITS, radius=FP_RADIUS, return_invalid=True
)
fingerprints, log = clean_na_and_duplicates(fingerprints, log)
fingerprints      = merge_ID_response_variable(fingerprints, df)

## 5. Activity labelling, train/test split and feature selection

In [ ]:
clean_desc = create_binary_activity_median(descriptors)
clean_fp   = create_binary_activity_median(fingerprints)

Xtr_d, Xte_d, ytr_d, yte_d = split_train_test(clean_desc, "Activity", test_size=TEST_SIZE, random_state=RANDOM_STATE)
Xtr_f, Xte_f, ytr_f, yte_f = split_train_test(clean_fp,   "Activity", test_size=TEST_SIZE, random_state=RANDOM_STATE)

Xtr_d, Xte_d, _, selected_features, _ = feature_selection_lasso(Xtr_d, ytr_d, Xte_d)
save_descriptor_list(selected_features, TARGET_KEY, "descriptors")

## 6. Model evaluation — Descriptors

In [ ]:
res_lr,  fig_lr,  _ = evaluate_logistic_regression(Xtr_d, ytr_d, Xte_d, yte_d)
res_knn, fig_knn, _ = evaluate_knn(Xtr_d, ytr_d, Xte_d, yte_d)
res_gnb, fig_gnb, _ = evaluate_gaussian_nb(Xtr_d, ytr_d, Xte_d, yte_d)
res_gb,  fig_gb,  _ = evaluate_gradient_boosting(Xtr_d, ytr_d, Xte_d, yte_d)
res_svm, fig_svm, _ = evaluate_svm(Xtr_d, ytr_d, Xte_d, yte_d)
res_rf,  fig_rf,  _ = evaluate_random_forest(Xtr_d, ytr_d, Xte_d, yte_d)
res_ann, fig_ann, _ = evaluate_mlp(Xtr_d, ytr_d, Xte_d, yte_d)

### Save results

In [ ]:
results_desc = {"LR": res_lr, "KNN": res_knn, "GNB": res_gnb,
               "GB": res_gb, "SVM": res_svm, "RF": res_rf, "ANN": res_ann}
figures_desc = {"LR": fig_lr, "KNN": fig_knn, "GNB": fig_gnb,
               "GB": fig_gb, "SVM": fig_svm, "RF": fig_rf, "ANN": fig_ann}
save_results(results_desc, figures_desc, TARGET_KEY, "descriptors")

## 7. Model evaluation — Fingerprints

In [ ]:
res_lr_f,  fig_lr_f,  _ = evaluate_logistic_regression(Xtr_f, ytr_f, Xte_f, yte_f)
res_knn_f, fig_knn_f, _ = evaluate_knn(Xtr_f, ytr_f, Xte_f, yte_f)
res_bnb_f, fig_bnb_f, _ = evaluate_bernoulli_nb(Xtr_f, ytr_f, Xte_f, yte_f)
res_gb_f,  fig_gb_f,  _ = evaluate_gradient_boosting(Xtr_f, ytr_f, Xte_f, yte_f)
res_svm_f, fig_svm_f, _ = evaluate_svm(Xtr_f, ytr_f, Xte_f, yte_f)
res_rf_f,  fig_rf_f,  _ = evaluate_random_forest(Xtr_f, ytr_f, Xte_f, yte_f)
res_ann_f, fig_ann_f, _ = evaluate_mlp(Xtr_f, ytr_f, Xte_f, yte_f)

### Save results

In [ ]:
results_fp = {"LR": res_lr_f, "KNN": res_knn_f, "BNB": res_bnb_f,
             "GB": res_gb_f,  "SVM": res_svm_f,  "RF": res_rf_f, "ANN": res_ann_f}
figures_fp = {"LR": fig_lr_f, "KNN": fig_knn_f, "BNB": fig_bnb_f,
             "GB": fig_gb_f,  "SVM": fig_svm_f,  "RF": fig_rf_f, "ANN": fig_ann_f}
save_results(results_fp, figures_fp, TARGET_KEY, "fingerprints")